In [1]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
kagglehub.competition_download(
    'kmu-rec-sys-26-rating-prediction',
    output_dir = "dataset"
)

100%|██████████| 5.76M/5.76M [00:01<00:00, 4.32MB/s]

Extracting files...


'dataset'

In [ ]:
!head dataset/train_small.csv

userId,movieId,rating,date
64777,16043,5.0,1997-07-01
64777,5112,2.0,1997-07-02
64777,24688,4.0,1997-07-02
64777,16639,4.0,1997-07-01
64777,13084,4.0,1997-07-02
64777,15142,4.0,1997-07-03
64777,5741,3.0,1997-07-03
64777,25503,3.0,1997-07-03
64777,2780,4.0,1997-07-01


In [ ]:
import csv
import torch
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

userIds, itemIds = {}, {}
n_users, n_items, n_ratings = 0,0,0
users, items, ratings = [], [], []

with open('dataset/train_small.csv',"r") as f:
  reader = csv.reader(f)
  next(reader)
  for userId, itemId, rating, _ in reader:
    if userId not in userIds:
      userIds[userId] = n_users
      n_users+=1
    if itemId not in itemIds:
      itemIds[itemId] = n_items
      n_items+=1

    users.append(userIds[userId])
    items.append(itemIds[itemId])
    ratings.append(float(rating))

users = torch.tensor(users).to(device)
items = torch.tensor(items).to(device)
ratings = torch.tensor(ratings).to(device)

In [ ]:
userIds

{'64777': 0,
 '42001': 1,
 '102026': 2,
 '67127': 3,
 '34856': 4,
 '78087': 5,
 '63515': 6,
 '55254': 7,
 '29481': 8,
 '66347': 9,
 '18419': 10,
 '50858': 11,
 '11811': 12,
 '96496': 13,
 '23190': 14,
 '56729': 15,
 '107340': 16,
 '19847': 17,
 '24242': 18,
 '76122': 19,
 '58725': 20,
 '57323': 21,
 '82736': 22,
 '50285': 23,
 '98493': 24,
 '35054': 25,
 '73488': 26,
 '98953': 27,
 '23618': 28,
 '66385': 29,
 '92582': 30,
 '3731': 31,
 '110593': 32,
 '53426': 33,
 '36109': 34,
 '99045': 35,
 '615': 36,
 '46208': 37,
 '2289': 38,
 '4330': 39,
 '114191': 40,
 '26335': 41,
 '56086': 42,
 '88416': 43,
 '74275': 44,
 '86958': 45,
 '106052': 46,
 '24783': 47,
 '31260': 48,
 '74025': 49,
 '76023': 50,
 '28298': 51,
 '84024': 52,
 '47441': 53,
 '108842': 54,
 '6490': 55,
 '8342': 56,
 '74832': 57,
 '66677': 58,
 '110583': 59,
 '89127': 60,
 '76506': 61,
 '26900': 62,
 '68551': 63,
 '46856': 64,
 '33083': 65,
 '13765': 66,
 '88480': 67,
 '1935': 68,
 '30011': 69,
 '22395': 70,
 '11936': 71,
 '1

In [ ]:
n_train = int(len(ratings)*0.9)
indices = torch.randperm(len(ratings)) # 데이터 섞기

train_indices = indices[:n_train]
valid_indices = indices[n_train:]

users_valid = users[valid_indices]
items_valid = items[valid_indices]
ratings_valid = ratings[valid_indices]

users = users[train_indices]
items = items[train_indices]
ratings = ratings[train_indices]

users_all = users.clone()
items_all = items.clone()
ratings_all = ratings.clone()

In [ ]:
lmd = 15
rank = 10 # 벡터의 차원
lr = 0.01
# mean + user_bias + item_bias + user_emb*item_emb
mean = ratings.mean().to(device)

user_bias = torch.zeros(n_users, requires_grad=True, device=device)
item_bias = torch.zeros(n_items, requires_grad = True, device=device)

user_emb = torch.normal(mean=0, std=0.01, size=(n_users,rank), requires_grad=True, device=device)
item_emb = torch.normal(mean=0, std=0.01, size=(n_items,rank), requires_grad=True, device=device)
optim = torch.optim.Adam([user_bias, item_bias, user_emb, item_emb],lr=lr)



In [ ]:
a = torch.tensor([9,8,7,6,5])

idx = torch.tensor([0,2,2,1,3,4])

a[idx]

tensor([9, 7, 7, 8, 6, 5])

In [ ]:
best_val = float('inf')
best_epoch = 0

for epoch in range(1000):

    h = mean + user_bias[users] + item_bias[items] + \
    torch.sum(user_emb[users] * item_emb[items], dim=1)

    sse = ((h - ratings)**2).sum()
    reg = lmd * ((user_bias**2).sum() + (item_bias**2).sum() +
                 (user_emb**2).sum() + (item_emb**2).sum())

    cost = sse + reg

    optim.zero_grad()
    cost.backward()
    optim.step()

    # validation
    if epoch % 50 == 0:
        with torch.no_grad():
            h_val = mean + user_bias[users_valid] + item_bias[items_valid] + \
                    torch.sum(user_emb[users_valid] * item_emb[items_valid], dim=1)

            mse_val = ((h_val - ratings_valid)**2).mean().item()

            print(f"epoch:{epoch}, val:{mse_val:.4f}")

            if mse_val < best_val:
                best_val = mse_val
                best_epoch = epoch

print(f"BEST EPOCH: {best_epoch}")

epoch:0, val:1.0907
epoch:50, val:0.6963
epoch:100, val:0.6622
epoch:150, val:0.6499
epoch:200, val:0.6450
epoch:250, val:0.6429
epoch:300, val:0.6421
epoch:350, val:0.6418
epoch:400, val:0.6417
epoch:450, val:0.6417
epoch:500, val:0.6417
epoch:550, val:0.6418
epoch:600, val:0.6419
epoch:650, val:0.6419
epoch:700, val:0.6420
epoch:750, val:0.6421
epoch:800, val:0.6422
epoch:850, val:0.6422
epoch:900, val:0.6423
epoch:950, val:0.6423
BEST EPOCH: 450


In [ ]:
# 전체 데이터로 다시 학습

mean = ratings_all.mean()

user_bias = torch.nn.Parameter(torch.zeros(n_users, device=device))
item_bias = torch.nn.Parameter(torch.zeros(n_items, device=device))

user_emb = torch.nn.Parameter(torch.randn(n_users, rank, device=device) * 0.01)
item_emb = torch.nn.Parameter(torch.randn(n_items, rank, device=device) * 0.01)

optim = torch.optim.Adam([user_bias, item_bias, user_emb, item_emb], lr=lr)

# best_epoch 만큼 학습
for epoch in range(best_epoch):

    h = mean + user_bias[users_all] + item_bias[items_all] + \
        torch.sum(user_emb[users_all] * item_emb[items_all], dim=1)

    sse = ((h - ratings_all)**2).sum()
    reg = lmd * ((user_bias**2).sum() + (item_bias**2).sum() +
                 (user_emb**2).sum() + (item_emb**2).sum())

    cost = sse + reg

    optim.zero_grad()
    cost.backward()
    optim.step()

    if epoch % 100 == 0:
        mse = (sse / len(ratings_all)).item()
        print(f"epoch:{epoch}, mse:{mse:.4f}")

epoch:0, mse:1.1047
epoch:100, mse:0.5679
epoch:200, mse:0.5358
epoch:300, mse:0.5319
epoch:400, mse:0.5308


In [ ]:
with open('dataset/test_small.csv') as fin, \
     open('solution.csv','w', newline='') as fout:

     reader = csv.reader(fin)
     next(reader)

     writer = csv.writer(fout)
     writer.writerow(["id", "rating"])
     for userId, itemId in reader:
      uid = userIds[userId]
      iid = itemIds[itemId]
      pred = mean + user_bias[uid] + item_bias[iid] + \
      (user_emb[uid]*item_emb[iid]).sum()
      pred = min(5, max(0.5, pred.item()))
      writer.writerow([f"{userId}_{itemId}", pred])
